# Product Review Sentiment Classification with LSTM

This notebook builds a binary sentiment classifier for product reviews using an LSTM neural network.

**Workflow:** load reviews → clean text → create sentiment labels → split data → fit tokenizer on training text only → train LSTM → evaluate on unseen test data.

> This is an educational experiment. The notebook uses a configurable sample of 500 reviews by default. Set `SAMPLE_SIZE = None` to use the full CSV (subject to available memory and training time).

## 1. Imports and configuration

Place `Reviews_sample.csv` beside this notebook for the reproducible lightweight version (recommended). You can create it from the original `Reviews.csv` using the included `prepare_dataset.py` script. The notebook also accepts the full `Reviews.csv` file if available.

In [ ]:
from pathlib import Path
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Prefer the small, committed sample; otherwise use the original full CSV.
SAMPLE_PATH = Path("Reviews_sample.csv")
FULL_DATA_PATH = Path("Reviews.csv")
DATA_PATH = SAMPLE_PATH if SAMPLE_PATH.exists() else FULL_DATA_PATH

MAX_WORDS = 5000
MAX_LENGTH = 100
EMBEDDING_DIM = 32
LSTM_UNITS = 32
EPOCHS = 5
BATCH_SIZE = 32

## 2. Load and validate the dataset

The repository includes a small sample file named `Reviews_sample.csv` (once added) so the notebook can run without downloading the full dataset. To reproduce the sample from the original Amazon Fine Food Reviews dataset, download `Reviews.csv` from [Kaggle](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews) and run the included `prepare_dataset.py` script. The original dataset contains 568,454 reviews and is approximately 300 MB, so it should not be committed directly to a normal Git repository.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Upload Reviews_sample.csv beside this notebook, "
        "or download the original Reviews.csv and run prepare_dataset.py. "
        "See README.md for step-by-step instructions."
    )

print(f"Loading dataset: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)

required_columns = {"Text", "Score"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

df = df[["Text", "Score"]].dropna().copy()
df["Text"] = df["Text"].astype(str)
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df = df.dropna(subset=["Score"])
df = df[df["Score"].between(1, 5)].copy()

if df.empty:
    raise ValueError("No valid reviews were found in the selected dataset.")

print(f"Number of reviews loaded: {len(df)}")
display(df.head())

## 3. Create binary sentiment labels

Ratings of 4 or 5 are labeled **positive (1)**. Ratings of 1, 2, or 3 are labeled **negative (0)**. This is the same labeling rule used in the original experiment.

In [ ]:
df["sentiment"] = (df["Score"] > 3).astype(int)

print(df["sentiment"].value_counts().rename(index={0: "Negative", 1: "Positive"}))
sns.countplot(data=df, x="sentiment", hue="sentiment", palette=["#d9534f", "#5cb85c"], legend=False)
plt.xticks([0, 1], ["Negative", "Positive"])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of reviews")
plt.show()

## 4. Clean review text

Text is lowercased, non-letter characters are removed, and common English stop words are filtered. Missing text values were removed during data validation.

In [ ]:
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    """Normalize a review and remove common English stop words."""
    text = text.lower()
    words = re.findall(r"[a-z]+", text)
    return " ".join(word for word in words if word not in stop_words)

df["clean_text"] = df["Text"].apply(clean_text)
df = df[df["clean_text"].str.strip().ne("")].copy()

if df["sentiment"].nunique() < 2:
    raise ValueError(
        "The selected data contains only one sentiment class. "
        "Use a larger or more representative dataset."
    )

display(df[["Text", "clean_text", "Score", "sentiment"]].head())

## 5. Split before tokenization

We split the raw review text into training and test sets **before fitting the tokenizer**. The tokenizer is fitted only on training text, preventing information from the test set from leaking into vocabulary construction. Stratification preserves the class proportions where possible.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"].to_numpy(),
    df["sentiment"].to_numpy(),
    test_size=0.20,
    random_state=SEED,
    stratify=df["sentiment"]
)

print(f"Training reviews: {len(X_train_text)}")
print(f"Test reviews: {len(X_test_text)}")

## 6. Tokenize and pad sequences

The vocabulary is learned from the training reviews only. Unknown words are represented by an out-of-vocabulary token. Each review is padded or truncated to `MAX_LENGTH` tokens.

In [ ]:
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding="pre", truncating="pre")
X_test = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding="pre", truncating="pre")

y_train = np.asarray(y_train, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

print(f"Training tensor shape: {X_train.shape}")
print(f"Test tensor shape: {X_test.shape}")
print(f"Configured vocabulary size: {MAX_WORDS}")

## 7. Build the LSTM model

- **Embedding:** converts token IDs into dense word vectors.
- **LSTM:** learns sequential patterns in the review.
- **Dropout:** reduces overfitting by randomly dropping some activations during training.
- **Sigmoid output:** returns a probability for the positive class.

In [ ]:
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, mask_zero=True),
    LSTM(LSTM_UNITS),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 8. Train the model

A validation split is taken from the training data. The test set is not used during training or model selection; it is reserved for the final evaluation.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=1
)

## 9. Evaluate on the held-out test set

The test set is evaluated once after training. Accuracy alone can hide class imbalance, so we also report precision, recall, F1-score, and a confusion matrix.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

probabilities = model.predict(X_test, verbose=0).ravel()
y_pred = (probabilities >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

## 10. Training curves

Compare training and validation accuracy/loss to look for signs of overfitting. The validation metrics are calculated from a portion of the training data, not the final test set.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 11. Predict sentiment for a new review

This helper reuses the tokenizer fitted on the training set and applies the same cleaning and padding steps.

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="pre",
        truncating="pre"
    )
    positive_probability = float(model.predict(padded, verbose=0)[0][0])
    label = "Positive" if positive_probability >= 0.5 else "Negative"
    return {
        "sentiment": label,
        "positive_probability": round(positive_probability, 4)
    }

predict_sentiment("The product quality is excellent and I am very happy with it.")

## 12. Limitations and next steps

- The default experiment uses only 500 reviews, which is too small to support broad performance claims. Set `SAMPLE_SIZE = None` to train on all valid rows.
- The rating-to-sentiment rule groups 3-star reviews with negative reviews; this is a modeling choice, not an objective truth.
- Results can vary with the selected sample, class distribution, hardware, and TensorFlow version.
- Further improvements could include a larger dataset, hyperparameter tuning, class-imbalance handling, early stopping, and comparison with a pretrained language model.
- The notebook does not save a trained model or tokenizer by default.